In [2]:
import pandas as pd

# -----------------------------
# Configuration
# -----------------------------
PAYSIM_PATH = "PS_20174392719_1491204439457_log.csv"   # Change if needed
NUM_CUSTOMERS = 8000
OUTPUT_FILE = "bank_transactions.csv"
RANDOM_STATE = 42

# -----------------------------
# Read PaySim
# -----------------------------
print("Reading PaySim dataset...")

df = pd.read_csv('paysim dataset.csv')

print(f"Total Transactions : {len(df):,}")

# -----------------------------
# Find all customer accounts
# -----------------------------
customers = df["nameOrig"].unique()

print(f"Unique Customers : {len(customers):,}")

# -----------------------------
# Randomly choose customers
# -----------------------------
selected_customers = (
    pd.Series(customers)
    .sample(NUM_CUSTOMERS, random_state=RANDOM_STATE)
)

print(f"Selected Customers : {len(selected_customers):,}")

# -----------------------------
# Keep all transactions
# -----------------------------
bank_df = df[df["nameOrig"].isin(selected_customers)].copy()

print(f"Transactions After Filtering : {len(bank_df):,}")

# -----------------------------
# Save
# -----------------------------
bank_df.to_csv(OUTPUT_FILE, index=False)

print("\nDone.")
print(f"Saved to {OUTPUT_FILE}")

Reading PaySim dataset...
Total Transactions : 6,362,620
Unique Customers : 6,353,307
Selected Customers : 8,000
Transactions After Filtering : 8,009

Done.
Saved to bank_transactions.csv


# Generating our new Dataset:

In [3]:
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)

In [6]:
CHUNK_SIZE = 100000

sampled_chunks = []

columns = [
    "step",
    "type",
    "amount",
    "nameOrig",
    "oldbalanceOrg",
    "newbalanceOrig",
    "nameDest",
    "oldbalanceDest",
    "newbalanceDest",
    "isFraud"
]

for chunk in pd.read_csv(
    "paysim dataset.csv",
    usecols=columns,
    chunksize=CHUNK_SIZE
):
    sampled = chunk.sample(
        frac=0.02,
        random_state=42
    )

    sampled_chunks.append(sampled)

bank_df = pd.concat(
    sampled_chunks,
    ignore_index=True
)

print(bank_df.shape)

(127252, 10)


In [7]:
bank_df = bank_df.sample(
    n=100000,
    random_state=42
).reset_index(drop=True)

print(bank_df.shape)

(100000, 10)


In [8]:
bank_df.to_csv(
    "bank_transactions_raw.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully


## Refining the new csv:

In [9]:
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)

bank_df = pd.read_csv("bank_transactions_raw.csv")

print(bank_df.shape)

bank_df.head()


(100000, 10)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud
0,254,PAYMENT,3853.04,C1469510588,0.0,0.00,M767993324,0.00,0.00,0
1,302,CASH_IN,278263.77,C1394025052,204530.0,482793.77,C389961354,300561.56,22297.78,0
2,39,CASH_OUT,328073.84,C2032213545,15865.0,0.00,C1021427324,2361146.10,2689219.94,0
3,253,TRANSFER,85912.64,C1138075457,0.0,0.00,C150003255,1555431.93,1641344.57,0
4,185,PAYMENT,17991.22,C1158304344,0.0,0.00,M1365982848,0.00,0.00,0


In [10]:
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)

bank_df = pd.read_csv("bank_transactions_raw.csv")

print(bank_df.shape)

bank_df.head()

(100000, 10)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud
0,254,PAYMENT,3853.04,C1469510588,0.0,0.00,M767993324,0.00,0.00,0
1,302,CASH_IN,278263.77,C1394025052,204530.0,482793.77,C389961354,300561.56,22297.78,0
2,39,CASH_OUT,328073.84,C2032213545,15865.0,0.00,C1021427324,2361146.10,2689219.94,0
3,253,TRANSFER,85912.64,C1138075457,0.0,0.00,C150003255,1555431.93,1641344.57,0
4,185,PAYMENT,17991.22,C1158304344,0.0,0.00,M1365982848,0.00,0.00,0


In [15]:
print(bank_df.columns.tolist())

['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud']


In [16]:
sender_ids = set(bank_df["nameOrig"].unique())

receiver_ids = set(bank_df["nameDest"].unique())

all_customers = list(sender_ids.union(receiver_ids))

print("Unique Sender IDs :", len(sender_ids))
print("Unique Receiver IDs :", len(receiver_ids))
print("Total Unique Customers :", len(all_customers))

Unique Sender IDs : 99998
Unique Receiver IDs : 92816
Total Unique Customers : 192810


In [17]:
customer_master = pd.DataFrame({
    "Paysim_ID": all_customers
})

customer_master.head()

,Paysim_ID
0,C980470768
1,C1661072942
2,C1278584897
3,C1504785763
4,C104905172


In [18]:
customer_master["Customer_ID"] = [
    f"CUST{i:07d}"
    for i in range(1, len(customer_master) + 1)
]

customer_master.head()

,Paysim_ID,Customer_ID
0,C980470768,CUST0000001
1,C1661072942,CUST0000002
2,C1278584897,CUST0000003
3,C1504785763,CUST0000004
4,C104905172,CUST0000005


In [19]:
import string

In [20]:
banks = [
    "State Bank of India",
    "HDFC Bank",
    "ICICI Bank",
    "Axis Bank",
    "Punjab National Bank",
    "Bank of Baroda",
    "Canara Bank",
    "Union Bank of India",
    "Kotak Mahindra Bank",
    "IndusInd Bank"
]

In [21]:
ifsc_codes = {
    "State Bank of India":"SBIN",
    "HDFC Bank":"HDFC",
    "ICICI Bank":"ICIC",
    "Axis Bank":"UTIB",
    "Punjab National Bank":"PUNB",
    "Bank of Baroda":"BARB",
    "Canara Bank":"CNRB",
    "Union Bank of India":"UBIN",
    "Kotak Mahindra Bank":"KKBK",
    "IndusInd Bank":"INDB"
}

In [22]:
cities = [
    "Surat",
    "Ahmedabad",
    "Mumbai",
    "Delhi",
    "Pune",
    "Jaipur",
    "Lucknow",
    "Indore",
    "Bhopal",
    "Nagpur",
    "Hyderabad",
    "Bengaluru",
    "Chennai",
    "Kolkata",
    "Patna"
]

In [23]:
customer_master["Bank_Name"] = np.random.choice(
    banks,
    len(customer_master)
)

customer_master.head()

,Paysim_ID,Customer_ID,Bank_Name
0,C980470768,CUST0000001,Canara Bank
1,C1661072942,CUST0000002,Axis Bank
2,C1278584897,CUST0000003,Union Bank of India
3,C1504785763,CUST0000004,Punjab National Bank
4,C104905172,CUST0000005,Canara Bank


In [24]:
customer_master["City"] = np.random.choice(
    cities,
    len(customer_master)
)

customer_master.head()

,Paysim_ID,Customer_ID,Bank_Name,City
0,C980470768,CUST0000001,Canara Bank,Bhopal
1,C1661072942,CUST0000002,Axis Bank,Pune
2,C1278584897,CUST0000003,Union Bank of India,Chennai
3,C1504785763,CUST0000004,Punjab National Bank,Mumbai
4,C104905172,CUST0000005,Canara Bank,Kolkata


In [25]:
customer_master["Account_Number"] = [
    str(random.randint(10**11,10**12-1))
    for _ in range(len(customer_master))
]

In [26]:
customer_master["IFSC"] = customer_master["Bank_Name"].apply(
    lambda x:
    ifsc_codes[x] + str(random.randint(1000000,9999999))
)

In [27]:
customer_master["Phone_Number"] = [
    str(random.randint(6000000000,9999999999))
    for _ in range(len(customer_master))
]

In [28]:
providers = [
    "oksbi",
    "okhdfcbank",
    "okicici",
    "ybl",
    "ibl"
]

customer_master["UPI_ID"] = customer_master.apply(
    lambda row:
    row["Phone_Number"] +
    "@" +
    random.choice(providers),
    axis=1
)

In [29]:
customer_master.head()

,Paysim_ID,Customer_ID,Bank_Name,City,Account_Number,IFSC,Phone_Number,UPI_ID
0,C980470768,CUST0000001,Canara Bank,Bhopal,223005401501,CNRB4034684,9876137445,9876137445@okhdfcbank
1,C1661072942,CUST0000002,Axis Bank,Pune,911856239313,UTIB2847846,7675203574,7675203574@oksbi
2,C1278584897,CUST0000003,Union Bank of India,Chennai,367469214295,UBIN7124901,9802299081,9802299081@okhdfcbank
3,C1504785763,CUST0000004,Punjab National Bank,Mumbai,251282538206,PUNB6347579,8307012433,8307012433@ibl
4,C104905172,CUST0000005,Canara Bank,Kolkata,214832269481,CNRB2946693,6539439394,6539439394@ibl


In [31]:
male_names = [
    "Aarav", "Aaryan", "Aashish", "Abhay", "Abhinav", "Abhishek", "Aditya",
    "Advait", "Ajay", "Akash", "Akhil", "Akshay", "Amit", "Amol", "Anand",
    "Anil", "Ankit", "Ansh", "Anupam", "Anurag", "Aravind", "Arjun", "Arnav",
    "Arun", "Arvind", "Aryan", "Ashok", "Ashwin", "Atharva", "Avinash", "Ayush",
    "Badrinath", "Balaji", "Bharat", "Bhaskar", "Bhuvan", "Bipin", "Brahma",
    "Chaitanya", "Chandan", "Chandra", "Charan", "Chetan", "Chirag", "Chitta",
    "Daksh", "Darshan", "Deepak", "Dev", "Devansh", "Deven", "Dhruv", "Dilip",
    "Dinesh", "Dipak", "Divyansh", "Eeshwar", "Ekansh", "Eklavya", "Farhan",
    "Feroz", "Gagan", "Ganesh", "Gaurav", "Gautam", "Girish", "Gopal",
    "Govind", "Gulshan", "Hardik", "Hari", "Harish", "Harsh", "Harshad",
    "Harshil", "Hemant", "Himesh", "Hitesh", "Hrithik", "Ishaan", "Ishwar",
    "Imran", "Inder", "Jagadish", "Jai", "Jaideep", "Jatin", "Jay", "Jayant",
    "Jeevan", "Jignesh", "Jitesh", "Jyotir", "Kabir", "Kailash", "Kalyan",
    "Kamal", "Karan", "Karthik", "Kartik", "Kaushal", "Kaushik", "Kavin",
    "Kedar", "Keshav", "Ketan", "Kiran", "Kishore", "Krishna", "Krunal",
    "Kumar", "Kunal", "Kush", "Lakshya", "Lalit", "Lokesh", "Luv", "Madhav",
    "Mahesh", "Manav", "Manish", "Manoj", "Mayank", "Meet", "Mehul", "Mihir",
    "Milan", "Milind", "Mitul", "Mohan", "Mohit", "Mukesh", "Nachiket",
    "Nakul", "Naman", "Narayan", "Narendra", "Naresh", "Naveen", "Navin",
    "Neeraj", "Nikhil", "Nikunj", "Nilesh", "Niraj", "Nirav", "Nirmal",
    "Nishant", "Nitin", "Om", "Omkar", "Ojas", "Pankaj", "Parag", "Param",
    "Parth", "Pavan", "Piyush", "Prabhat", "Prabhu", "Pradeep", "Pranav",
    "Pranay", "Pratik", "Praveen", "Prem", "Pritam", "Puneet", "Pushkar",
    "Rachit", "Raghav", "Rahul", "Raj", "Rajan", "Rajat", "Rajeev", "Rajesh",
    "Rakesh", "Ram", "Raman", "Ramesh", "Randhir", "Ravi", "Ravindra",
    "Reyansh", "Rishi", "Ritik", "Rohan", "Rohit", "Roshan", "Rupesh",
    "Sachin", "Sagar", "Sahil", "Samir", "Sanjay", "Sanjeev", "Sanket",
    "Sarthak", "Saurabh"
]

In [32]:
female_names = [
    "Aadhya", "Aalia", "Aanya", "Aaradhya", "Aarthi", "Aashi", "Aayushi", "Aditi", "Advika", "Aishwarya",
    "Akanksha", "Akhila", "Akshara", "Alia", "Amira", "Amrita", "Ananya", "Anika", "Anita", "Anjali",
    "Ankita", "Anshika", "Anushka", "Aparna", "Aradhana", "Arpita", "Asha", "Ashwini", "Avani", "Ayushi",
    "Bhavana", "Bhavani", "Bhavya", "Bhoomi", "Bipasha", "Chaitali", "Chandani", "Charu", "Chetana", "Chhavi",
    "Daksha", "Damini", "Darshana", "Deepa", "Deepali", "Deepika", "Deepti", "Devanshi", "Devika", "Dhanya",
    "Dhara", "Drishti", "Esha", "Ekta", "Falguni", "Farah", "Gargi", "Garima", "Gauri", "Gayatri",
    "Geeta", "Geetanjali", "Gunjan", "Harshita", "Hema", "Hemalatha", "Himani", "Indira", "Indu", "Isha",
    "Ishani", "Ishika", "Ishita", "Jagruti", "Jahnavi", "Janki", "Jaya", "Jayashree", "Jia", "Juhi",
    "Jyoti", "Jyotsna", "Kajal", "Kalpana", "Kalyani", "Kamala", "Kanika", "Kareena", "Karishma", "Kavita",
    "Kavya", "Khushi", "Kiran", "Kirti", "Komal", "Kriti", "Krutika", "Kusum", "Lata", "Latika",
    "Laxmi", "Leela", "Madhuri", "Mahima", "Maithili", "Malavika", "Malini", "Mamata", "Manasi", "Manisha",
    "Manju", "Meena", "Meenakshi", "Meera", "Megha", "Meghna", "Mitali", "Mohini", "Monica", "Mrinali",
    "Nalini", "Namita", "Namrata", "Nandini", "Nandita", "Navya", "Neelam", "Neha", "Nidhi", "Niharika",
    "Nikita", "Nilima", "Nisha", "Nita", "Nivedita", "Niyati", "Nupur", "Nutan", "Ojaswini", "Padma",
    "Pallavi", "Pari", "Parvati", "Payal", "Pooja", "Poonam", "Poornima", "Prachi", "Pragya", "Prerana",
    "Priti", "Priya", "Priyanka", "Rachana", "Radha", "Radhika", "Ragini", "Rajani", "Rakhi", "Ramya",
    "Rani", "Rashmi", "Raveena", "Rekha", "Renu", "Renuka", "Reshma", "Richa", "Riddhi", "Riya",
    "Rohini", "Roshni", "Ruchi", "Rupali", "Saanvi", "Sachi", "Sadhana", "Sakshi", "Saloni", "Sangeeta",
    "Sanjana", "Sapna", "Saraswati", "Sarika", "Sarita", "Seema", "Shalini", "Shikha", "Shilpa", "Shreya",
    "Shruti", "Smriti", "Sneha", "Sonali", "Sonia", "Srishti", "Suman", "Sunita", "Sushma", "Swati"
]

In [33]:
last_names = [
"Sharma","Verma","Gupta","Patel","Shah","Bhatt","Mehta","Desai","Joshi","Pandya",
"Trivedi","Dave","Parmar","Chauhan","Solanki","Rathod","Yadav","Singh","Kaur","Gill",
"Sandhu","Kapoor","Malhotra","Arora","Bansal","Agarwal","Jain","Mittal","Saxena","Tiwari",
"Mishra","Pandey","Dubey","Tripathi","Chaturvedi","Srivastava","Roy","Das","Bose","Ghosh",
"Banerjee","Chatterjee","Mukherjee","Dutta","Reddy","Naidu","Rao","Nair","Menon","Iyer",
"Krishnan","Pillai","Subramanian","Kulkarni","Patil","Deshmukh","Pawar","Sawant","More","Shinde",
"Naik","Shetty","Hegde","Acharya","Bora","Kalita","Ali","Khan","Ansari","Qureshi",
"Shaikh","Pathan","Syed","Thomas","Mathew","Varghese","Fernandes","D'Souza","Pereira","Lal",
"Rawat","Negi","Bisht","Mahajan","Soni","Chawla","Kohli","Bhatia","Goel","Bagchi",
"Kar","Sen","Paul","Prasad","Sahu","Behera","Mohanty","Barik","Majumdar","Bhattacharya"
]

In [34]:
customer_master["Gender"] = np.random.choice(
    ["Male", "Female"],
    len(customer_master),
    p=[0.52, 0.48]
)

In [35]:
def generate_name(gender):

    if gender == "Male":
        first = random.choice(male_names)
    else:
        first = random.choice(female_names)

    last = random.choice(last_names)

    return f"{first} {last}"

customer_master["Customer_Name"] = customer_master["Gender"].apply(generate_name)

In [36]:
customer_master[["Customer_Name","Gender"]].sample(20)

,Customer_Name,Gender
66210,Divyansh Kalita,Male
45484,Rakhi Iyer,Female
182806,Riya Das,Female
36812,Renu Bisht,Female
173100,Sarika Kapoor,Female
173447,Anupam Sawant,Male
11666,Kumar Qureshi,Male
119444,Ishika Tiwari,Female
9656,Devika Tiwari,Female
151752,Kabir Bose,Male


In [37]:
email_domains = [
    "gmail.com",
    "yahoo.com",
    "outlook.com",
    "hotmail.com",
    "icloud.com"
]

customer_master["Email"] = customer_master.apply(
    lambda row:
    row["Customer_Name"].lower().replace(" ", ".")
    + str(random.randint(1,999))
    + "@"
    + random.choice(email_domains),
    axis=1
)

In [38]:
from datetime import datetime, timedelta

today = datetime.today()

dob = []

for _ in range(len(customer_master)):

    age = random.randint(18,75)

    days = random.randint(0,365)

    birth = today - timedelta(days=age*365+days)

    dob.append(birth.date())

customer_master["DOB"] = dob

In [39]:
joined = []

for _ in range(len(customer_master)):

    years = random.randint(1,15)

    days = random.randint(0,365)

    d = today - timedelta(days=years*365+days)

    joined.append(d.date())

customer_master["Customer_Since"] = joined

In [40]:
occupations = [

"Student",
"Engineer",
"Doctor",
"Teacher",
"Business",
"Lawyer",
"Farmer",
"Government Employee",
"Private Employee",
"Self Employed",
"Accountant",
"Sales Executive",
"Software Developer",
"Consultant",
"Retired",
"Police Officer",
"Professor",
"Shop Owner",
"Driver",
"Unemployed"

]

customer_master["Occupation"] = np.random.choice(
    occupations,
    len(customer_master)
)

In [41]:
customer_master["Annual_Income"] = np.random.randint(
    150000,
    4000000,
    len(customer_master)
)

In [42]:
customer_master["Annual_Income"] = np.random.randint(
    150000,
    4000000,
    len(customer_master)
)

In [43]:
customer_master["KYC_Status"] = np.random.choice(

["Verified","Pending"],

len(customer_master),

p=[0.98,0.02]

)

In [44]:
customer_master["Aadhaar"] = [

str(random.randint(100000000000,999999999999))

for _ in range(len(customer_master))

]

In [45]:
letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

customer_master["PAN"] = [

random.choice(letters)
+ random.choice(letters)
+ random.choice(letters)
+ random.choice(letters)
+ random.choice(letters)
+ str(random.randint(1000,9999))
+ random.choice(letters)

for _ in range(len(customer_master))

]

In [46]:
customer_master["Device_ID"] = [

"DEV"+str(random.randint(10000000,99999999))

for _ in range(len(customer_master))

]

In [47]:
customer_master["IMEI"] = [

str(random.randint(100000000000000,999999999999999))

for _ in range(len(customer_master))

]

In [48]:
customer_master["SIM_Number"] = [

str(random.randint(8900000000000000000,8999999999999999999))

for _ in range(len(customer_master))

]

In [52]:
customer_master.to_csv(

"customer_master.csv",

index=False

)

print(customer_master.shape)

customer_master.head()

(192810, 21)


,Paysim_ID,Customer_ID,Bank_Name,City,Account_Number,IFSC,Phone_Number,UPI_ID,Gender,Customer_Name,...,DOB,Customer_Since,Occupation,Annual_Income,KYC_Status,Aadhaar,PAN,Device_ID,IMEI,SIM_Number
0,C980470768,CUST0000001,Canara Bank,Bhopal,223005401501,CNRB4034684,9876137445,9876137445@okhdfcbank,Male,Niraj Lal,...,1979-01-15,2017-10-08,Engineer,3733963,Verified,592337894135,GYLNC8130S,DEV46180493,704849945366304,8908844093645845822
1,C1661072942,CUST0000002,Axis Bank,Pune,911856239313,UTIB2847846,7675203574,7675203574@oksbi,Male,Raghav Kaur,...,1962-11-01,2023-04-30,Business,1024034,Verified,804901053843,TJUFF8228U,DEV18471166,476901535879387,8971533799543255778
2,C1278584897,CUST0000003,Union Bank of India,Chennai,367469214295,UBIN7124901,9802299081,9802299081@okhdfcbank,Male,Eklavya Srivastava,...,1993-01-12,2011-09-27,Student,2396859,Verified,846416088312,TJAJK8210S,DEV88761645,313922826100062,8975770667602947445
3,C1504785763,CUST0000004,Punjab National Bank,Mumbai,251282538206,PUNB6347579,8307012433,8307012433@ibl,Female,Sunita Roy,...,1998-11-08,2010-09-11,Student,2355699,Verified,738059893920,CAUZP7040Z,DEV93987154,419365274232192,8906655297104633890
4,C104905172,CUST0000005,Canara Bank,Kolkata,214832269481,CNRB2946693,6539439394,6539439394@ibl,Male,Aashish Banerjee,...,1981-09-04,2024-11-18,Accountant,476100,Verified,276877127316,SENYG7140Z,DEV90720393,669835457460092,8993517761120241202


In [53]:
customer_master.to_csv(

"customer_master.csv",

index=False

)

print(customer_master.shape)

customer_master.head()

(192810, 21)


,Paysim_ID,Customer_ID,Bank_Name,City,Account_Number,IFSC,Phone_Number,UPI_ID,Gender,Customer_Name,...,DOB,Customer_Since,Occupation,Annual_Income,KYC_Status,Aadhaar,PAN,Device_ID,IMEI,SIM_Number
0,C980470768,CUST0000001,Canara Bank,Bhopal,223005401501,CNRB4034684,9876137445,9876137445@okhdfcbank,Male,Niraj Lal,...,1979-01-15,2017-10-08,Engineer,3733963,Verified,592337894135,GYLNC8130S,DEV46180493,704849945366304,8908844093645845822
1,C1661072942,CUST0000002,Axis Bank,Pune,911856239313,UTIB2847846,7675203574,7675203574@oksbi,Male,Raghav Kaur,...,1962-11-01,2023-04-30,Business,1024034,Verified,804901053843,TJUFF8228U,DEV18471166,476901535879387,8971533799543255778
2,C1278584897,CUST0000003,Union Bank of India,Chennai,367469214295,UBIN7124901,9802299081,9802299081@okhdfcbank,Male,Eklavya Srivastava,...,1993-01-12,2011-09-27,Student,2396859,Verified,846416088312,TJAJK8210S,DEV88761645,313922826100062,8975770667602947445
3,C1504785763,CUST0000004,Punjab National Bank,Mumbai,251282538206,PUNB6347579,8307012433,8307012433@ibl,Female,Sunita Roy,...,1998-11-08,2010-09-11,Student,2355699,Verified,738059893920,CAUZP7040Z,DEV93987154,419365274232192,8906655297104633890
4,C104905172,CUST0000005,Canara Bank,Kolkata,214832269481,CNRB2946693,6539439394,6539439394@ibl,Male,Aashish Banerjee,...,1981-09-04,2024-11-18,Accountant,476100,Verified,276877127316,SENYG7140Z,DEV90720393,669835457460092,8993517761120241202


In [55]:
customer_master = pd.read_csv("customer_master.csv")

In [56]:
customer_lookup = customer_master.set_index("Paysim_ID")

In [57]:
print(customer_lookup.shape)

customer_lookup.head()

(192810, 20)


,Customer_ID,Bank_Name,City,Account_Number,IFSC,Phone_Number,UPI_ID,Gender,Customer_Name,Email,DOB,Customer_Since,Occupation,Annual_Income,KYC_Status,Aadhaar,PAN,Device_ID,IMEI,SIM_Number
Paysim_ID,,,,,,,,,,,,,,,,,,,,
C980470768,CUST0000001,Canara Bank,Bhopal,223005401501,CNRB4034684,9876137445,9876137445@okhdfcbank,Male,Niraj Lal,niraj.lal636@outlook.com,1979-01-15,2017-10-08,Engineer,3733963,Verified,592337894135,GYLNC8130S,DEV46180493,704849945366304,8908844093645845822
C1661072942,CUST0000002,Axis Bank,Pune,911856239313,UTIB2847846,7675203574,7675203574@oksbi,Male,Raghav Kaur,raghav.kaur589@outlook.com,1962-11-01,2023-04-30,Business,1024034,Verified,804901053843,TJUFF8228U,DEV18471166,476901535879387,8971533799543255778
C1278584897,CUST0000003,Union Bank of India,Chennai,367469214295,UBIN7124901,9802299081,9802299081@okhdfcbank,Male,Eklavya Srivastava,eklavya.srivastava344@icloud.com,1993-01-12,2011-09-27,Student,2396859,Verified,846416088312,TJAJK8210S,DEV88761645,313922826100062,8975770667602947445
C1504785763,CUST0000004,Punjab National Bank,Mumbai,251282538206,PUNB6347579,8307012433,8307012433@ibl,Female,Sunita Roy,sunita.roy23@yahoo.com,1998-11-08,2010-09-11,Student,2355699,Verified,738059893920,CAUZP7040Z,DEV93987154,419365274232192,8906655297104633890
C104905172,CUST0000005,Canara Bank,Kolkata,214832269481,CNRB2946693,6539439394,6539439394@ibl,Male,Aashish Banerjee,aashish.banerjee847@hotmail.com,1981-09-04,2024-11-18,Accountant,476100,Verified,276877127316,SENYG7140Z,DEV90720393,669835457460092,8993517761120241202


In [58]:
customer_master.columns

Index(['Paysim_ID', 'Customer_ID', 'Bank_Name', 'City', 'Account_Number',
       'IFSC', 'Phone_Number', 'UPI_ID', 'Gender', 'Customer_Name', 'Email',
       'DOB', 'Customer_Since', 'Occupation', 'Annual_Income', 'KYC_Status',
       'Aadhaar', 'PAN', 'Device_ID', 'IMEI', 'SIM_Number'],
      dtype='object')

In [59]:
bank_transactions = bank_df.copy()

In [60]:
sender_master = customer_master.copy()

sender_master = sender_master.rename(
    columns={
        col: "Sender_" + col
        for col in sender_master.columns
        if col != "Paysim_ID"
    }
)

sender_master.head()

,Paysim_ID,Sender_Customer_ID,Sender_Bank_Name,Sender_City,Sender_Account_Number,Sender_IFSC,Sender_Phone_Number,Sender_UPI_ID,Sender_Gender,Sender_Customer_Name,...,Sender_DOB,Sender_Customer_Since,Sender_Occupation,Sender_Annual_Income,Sender_KYC_Status,Sender_Aadhaar,Sender_PAN,Sender_Device_ID,Sender_IMEI,Sender_SIM_Number
0,C980470768,CUST0000001,Canara Bank,Bhopal,223005401501,CNRB4034684,9876137445,9876137445@okhdfcbank,Male,Niraj Lal,...,1979-01-15,2017-10-08,Engineer,3733963,Verified,592337894135,GYLNC8130S,DEV46180493,704849945366304,8908844093645845822
1,C1661072942,CUST0000002,Axis Bank,Pune,911856239313,UTIB2847846,7675203574,7675203574@oksbi,Male,Raghav Kaur,...,1962-11-01,2023-04-30,Business,1024034,Verified,804901053843,TJUFF8228U,DEV18471166,476901535879387,8971533799543255778
2,C1278584897,CUST0000003,Union Bank of India,Chennai,367469214295,UBIN7124901,9802299081,9802299081@okhdfcbank,Male,Eklavya Srivastava,...,1993-01-12,2011-09-27,Student,2396859,Verified,846416088312,TJAJK8210S,DEV88761645,313922826100062,8975770667602947445
3,C1504785763,CUST0000004,Punjab National Bank,Mumbai,251282538206,PUNB6347579,8307012433,8307012433@ibl,Female,Sunita Roy,...,1998-11-08,2010-09-11,Student,2355699,Verified,738059893920,CAUZP7040Z,DEV93987154,419365274232192,8906655297104633890
4,C104905172,CUST0000005,Canara Bank,Kolkata,214832269481,CNRB2946693,6539439394,6539439394@ibl,Male,Aashish Banerjee,...,1981-09-04,2024-11-18,Accountant,476100,Verified,276877127316,SENYG7140Z,DEV90720393,669835457460092,8993517761120241202


In [61]:
bank_transactions = bank_transactions.merge(
    sender_master,
    left_on="nameOrig",
    right_on="Paysim_ID",
    how="left"
)

In [62]:
bank_transactions.drop(
    columns=["Paysim_ID"],
    inplace=True
)

In [63]:
receiver_master = customer_master.copy()

receiver_master = receiver_master.rename(
    columns={
        col: "Receiver_" + col
        for col in receiver_master.columns
        if col != "Paysim_ID"
    }
)

In [64]:
bank_transactions = bank_transactions.merge(
    receiver_master,
    left_on="nameDest",
    right_on="Paysim_ID",
    how="left"
)

In [65]:
bank_transactions.drop(
    columns=["Paysim_ID"],
    inplace=True
)

In [66]:
bank_transactions.drop(
    columns=[
        "nameOrig",
        "nameDest"
    ],
    inplace=True
)

In [67]:
print(bank_transactions.shape)

bank_transactions.head()

(100000, 48)


,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,Sender_Customer_ID,Sender_Bank_Name,...,Receiver_DOB,Receiver_Customer_Since,Receiver_Occupation,Receiver_Annual_Income,Receiver_KYC_Status,Receiver_Aadhaar,Receiver_PAN,Receiver_Device_ID,Receiver_IMEI,Receiver_SIM_Number
0,254,PAYMENT,3853.04,0.0,0.00,0.00,0.00,0,CUST0113495,Union Bank of India,...,1963-03-21,2022-02-14,Teacher,1021421,Verified,816635336286,PKBQG9404Z,DEV56845324,568888046000911,8983944259644222170
1,302,CASH_IN,278263.77,204530.0,482793.77,300561.56,22297.78,0,CUST0031344,Kotak Mahindra Bank,...,1953-09-12,2013-05-03,Government Employee,517058,Verified,483341868766,PKLMH1686P,DEV53795527,593797810114220,8934614614389157455
2,39,CASH_OUT,328073.84,15865.0,0.00,2361146.10,2689219.94,0,CUST0015822,IndusInd Bank,...,1956-04-14,2022-08-25,Business,1003837,Verified,334394509598,JRJIT2176Y,DEV51994243,187252823146913,8981764420326083390
3,253,TRANSFER,85912.64,0.0,0.00,1555431.93,1641344.57,0,CUST0161046,Punjab National Bank,...,1964-12-28,2020-12-21,Shop Owner,3287827,Verified,814005872718,QYOAV2042N,DEV76704016,532751016155847,8945607117900515515
4,185,PAYMENT,17991.22,0.0,0.00,0.00,0.00,0,CUST0023815,Punjab National Bank,...,1968-09-09,2013-06-27,Doctor,1252546,Verified,314920969212,VMILZ1009D,DEV50999971,571630952634620,8912066880424968199


In [68]:
bank_transactions["Transaction_ID"] = [
    f"TXN{str(i).zfill(10)}"
    for i in range(1, len(bank_transactions) + 1)
]

In [69]:
from datetime import datetime, timedelta

In [70]:
import random

start = datetime(2025, 1, 1)
end = datetime(2025, 12, 31, 23, 59, 59)

timestamps = []

for _ in range(len(bank_transactions)):

    seconds = random.randint(
        0,
        int((end - start).total_seconds())
    )

    timestamps.append(
        start + timedelta(seconds=seconds)
    )

bank_transactions["Timestamp"] = timestamps

In [71]:
bank_transactions = bank_transactions.sort_values(
    "Timestamp"
).reset_index(drop=True)

In [72]:
mode_map = {
    "PAYMENT": "UPI",
    "TRANSFER": "IMPS",
    "CASH_OUT": "ATM Withdrawal",
    "CASH_IN": "Cash Deposit",
    "DEBIT": "POS",
}

bank_transactions["Transaction_Mode"] = (
    bank_transactions["type"]
    .map(mode_map)
    .fillna("NEFT")
)

In [73]:
bank_transactions["Transaction_Status"] = np.random.choice(
    ["SUCCESS", "FAILED", "PENDING"],
    size=len(bank_transactions),
    p=[0.97, 0.02, 0.01]
)

In [74]:
bank_transactions["Reference_Number"] = [
    "UTR" + str(random.randint(100000000000, 999999999999))
    for _ in range(len(bank_transactions))
]

In [75]:
bank_transactions[[
    "Transaction_ID",
    "Timestamp",
    "Transaction_Mode",
    "Transaction_Status",
    "Reference_Number",
    "amount"
]].head()

,Transaction_ID,Timestamp,Transaction_Mode,Transaction_Status,Reference_Number,amount
0,TXN0000004306,2025-01-01 00:13:30,Cash Deposit,SUCCESS,UTR800662653430,262014.34
1,TXN0000027442,2025-01-01 00:19:08,UPI,SUCCESS,UTR227703190014,13464.49
2,TXN0000041957,2025-01-01 00:22:04,UPI,SUCCESS,UTR284025963296,6426.26
3,TXN0000017633,2025-01-01 00:32:04,Cash Deposit,SUCCESS,UTR255061396090,40435.27
4,TXN0000081101,2025-01-01 00:38:25,ATM Withdrawal,SUCCESS,UTR231296626607,558916.57


## Finalizing our dataset:

In [76]:
bank_transactions["Transaction_ID"] = [
    f"TXN{str(i).zfill(10)}"
    for i in range(1, len(bank_transactions)+1)
]

In [77]:
from datetime import datetime, timedelta
import random

In [78]:
start_date = datetime(2025,1,1)

end_date = datetime(2025,12,31,23,59,59)

In [79]:
timestamps=[]

for i in range(len(bank_transactions)):

    seconds=random.randint(
        0,
        int((end_date-start_date).total_seconds())
    )

    timestamps.append(
        start_date+timedelta(seconds=seconds)
    )

bank_transactions["Timestamp"]=timestamps

In [80]:
bank_transactions=bank_transactions.sort_values(
    "Timestamp"
).reset_index(drop=True)

In [81]:
bank_transactions["Transaction_ID"]=[
    f"TXN{str(i).zfill(10)}"
    for i in range(1,len(bank_transactions)+1)
]

In [82]:
mode_map={

"PAYMENT":"UPI",

"TRANSFER":"IMPS",

"CASH_OUT":"ATM",

"CASH_IN":"Cash Deposit",

"DEBIT":"POS"

}

In [83]:
bank_transactions["Transaction_Mode"]=(
    bank_transactions["type"]
    .map(mode_map)
    .fillna("NEFT")
)

In [84]:
bank_transactions["Transaction_Status"]=np.random.choice(

["SUCCESS","FAILED","PENDING"],

len(bank_transactions),

p=[0.97,0.02,0.01]

)

In [85]:
bank_transactions["Currency"]="INR"

In [86]:
bank_transactions["UTR_Number"]=[

"UTR"+str(random.randint(
100000000000,
999999999999))

for i in range(len(bank_transactions))

]

In [87]:
merchant_names=[

"Amazon",

"Flipkart",

"Myntra",

"Swiggy",

"Zomato",

"Blinkit",

"Reliance Fresh",

"DMart",

"BigBasket",

"IRCTC",

"BookMyShow",

"Uber",

"Ola",

"Indian Oil",

"HP Petrol",

"Bharat Petroleum",

"Apollo Pharmacy",

"MedPlus",

"Jio Recharge",

"Airtel Recharge",

"Electricity Board",

"Water Board",

"LIC",

"Income Tax",

"Municipal Corporation"

]

In [88]:
merchant_category={

"Amazon":"Shopping",

"Flipkart":"Shopping",

"Myntra":"Shopping",

"Swiggy":"Food",

"Zomato":"Food",

"Blinkit":"Groceries",

"Reliance Fresh":"Groceries",

"DMart":"Groceries",

"BigBasket":"Groceries",

"IRCTC":"Travel",

"BookMyShow":"Entertainment",

"Uber":"Transport",

"Ola":"Transport",

"Indian Oil":"Fuel",

"HP Petrol":"Fuel",

"Bharat Petroleum":"Fuel",

"Apollo Pharmacy":"Medical",

"MedPlus":"Medical",

"Jio Recharge":"Recharge",

"Airtel Recharge":"Recharge",

"Electricity Board":"Utilities",

"Water Board":"Utilities",

"LIC":"Insurance",

"Income Tax":"Government",

"Municipal Corporation":"Government"

}

In [89]:
merchants=[]

categories=[]

In [90]:
for mode in bank_transactions["Transaction_Mode"]:

    if mode=="UPI":

        m=random.choice(merchant_names)

    elif mode=="POS":

        m=random.choice(merchant_names)

    elif mode=="ATM":

        m="ATM Withdrawal"

    elif mode=="Cash Deposit":

        m="Cash Deposit"

    else:

        m=random.choice(merchant_names)

    merchants.append(m)

    categories.append(

        merchant_category.get(

            m,

            "Bank"

        )

    )

In [91]:
bank_transactions["Merchant_Name"]=merchants

bank_transactions["Merchant_Category"]=categories

In [92]:
bank_transactions.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,Sender_Customer_ID,Sender_Bank_Name,...,Receiver_SIM_Number,Transaction_ID,Timestamp,Transaction_Mode,Transaction_Status,Reference_Number,Currency,UTR_Number,Merchant_Name,Merchant_Category
0,637,CASH_OUT,12170.46,97084.15,84913.69,114770.19,126940.64,0,CUST0088000,IndusInd Bank,...,8930541142482852126,TXN0000000001,2025-01-01 00:01:21,ATM,SUCCESS,UTR344461950738,INR,UTR123873481368,ATM Withdrawal,Bank
1,162,CASH_IN,175774.54,5542.00,181316.54,172798.84,0.00,0,CUST0173395,Axis Bank,...,8999007851962565633,TXN0000000002,2025-01-01 00:01:39,Cash Deposit,SUCCESS,UTR630307748297,INR,UTR773663556037,Cash Deposit,Bank
2,301,CASH_IN,78682.34,5924793.94,6003476.28,233950.81,155268.47,0,CUST0044406,ICICI Bank,...,8992180879000828174,TXN0000000003,2025-01-01 00:03:42,Cash Deposit,SUCCESS,UTR982070254221,INR,UTR818079976787,Cash Deposit,Bank
3,210,TRANSFER,172184.82,0.00,0.00,1719669.27,1891854.09,0,CUST0103746,Union Bank of India,...,8937280074710967420,TXN0000000004,2025-01-01 00:12:03,IMPS,SUCCESS,UTR886808663023,INR,UTR980394273515,DMart,Groceries
4,253,TRANSFER,1838884.55,11176.00,0.00,1013524.75,2852409.30,0,CUST0105884,Union Bank of India,...,8964797583753955772,TXN0000000005,2025-01-01 00:17:55,IMPS,SUCCESS,UTR109933268220,INR,UTR572341019428,Bharat Petroleum,Fuel


In [93]:
bank_transactions.drop(
    columns=[
        "step",
        "oldbalanceOrg",
        "newbalanceOrig",
        "oldbalanceDest",
        "newbalanceDest",
        "type"
    ],
    inplace=True,
    errors="ignore"
)

In [94]:
bank_transactions = (
    bank_transactions
    .sort_values("Timestamp")
    .reset_index(drop=True)
)

In [95]:
bank_transactions["Transaction_ID"] = [
    f"TXN{str(i).zfill(10)}"
    for i in range(1, len(bank_transactions) + 1)
]

In [96]:
first_columns = [
    "Transaction_ID",
    "Timestamp",
    "UTR_Number",
    "Transaction_Mode",
    "Transaction_Status",
    "Currency",
    "amount"
]

remaining_columns = [
    c for c in bank_transactions.columns
    if c not in first_columns
]

bank_transactions = bank_transactions[
    first_columns + remaining_columns
]

In [97]:
bank_transactions.rename(
    columns={
        "amount": "Transaction_Amount"
    },
    inplace=True
)

In [98]:
print("Shape :", bank_transactions.shape)
bank_transactions.head()

Shape : (100000, 51)


,Transaction_ID,Timestamp,UTR_Number,Transaction_Mode,Transaction_Status,Currency,Transaction_Amount,isFraud,Sender_Customer_ID,Sender_Bank_Name,...,Receiver_Annual_Income,Receiver_KYC_Status,Receiver_Aadhaar,Receiver_PAN,Receiver_Device_ID,Receiver_IMEI,Receiver_SIM_Number,Reference_Number,Merchant_Name,Merchant_Category
0,TXN0000000001,2025-01-01 00:01:21,UTR123873481368,ATM,SUCCESS,INR,12170.46,0,CUST0088000,IndusInd Bank,...,1596319,Verified,212481665672,NHKNJ8323P,DEV44814735,222327911347456,8930541142482852126,UTR344461950738,ATM Withdrawal,Bank
1,TXN0000000002,2025-01-01 00:01:39,UTR773663556037,Cash Deposit,SUCCESS,INR,175774.54,0,CUST0173395,Axis Bank,...,907364,Verified,272336379965,NBEZX3931V,DEV96392357,969917165658087,8999007851962565633,UTR630307748297,Cash Deposit,Bank
2,TXN0000000003,2025-01-01 00:03:42,UTR818079976787,Cash Deposit,SUCCESS,INR,78682.34,0,CUST0044406,ICICI Bank,...,1362285,Verified,329188580711,UGRDD7623J,DEV48990617,121022249994773,8992180879000828174,UTR982070254221,Cash Deposit,Bank
3,TXN0000000004,2025-01-01 00:12:03,UTR980394273515,IMPS,SUCCESS,INR,172184.82,0,CUST0103746,Union Bank of India,...,3781561,Verified,278341675629,PCIKS6953U,DEV49560694,897359907056358,8937280074710967420,UTR886808663023,DMart,Groceries
4,TXN0000000005,2025-01-01 00:17:55,UTR572341019428,IMPS,SUCCESS,INR,1838884.55,0,CUST0105884,Union Bank of India,...,2209671,Verified,575968857380,HTIIJ1627W,DEV25315074,323603801377848,8964797583753955772,UTR109933268220,Bharat Petroleum,Fuel


In [1]:
bank_transactions.to_csv(
    "bank_transactions.csv",
    index=False
)

print("bank_transactions.csv saved successfully.")

NameError: name 'bank_transactions' is not defined

## Creating Fraud Transactions:

In [2]:
import pandas as pd
import numpy as np
import random

bank_transactions = pd.read_csv(
    "bank_transactions.csv",
    parse_dates=["Timestamp"]
)

bank_transactions.head()

,Transaction_ID,Timestamp,UTR_Number,Transaction_Mode,Transaction_Status,Currency,Transaction_Amount,Sender_Customer_ID,Sender_Bank_Name,Sender_City,...,Receiver_Annual_Income,Receiver_KYC_Status,Receiver_Aadhaar,Receiver_PAN,Receiver_Device_ID,Receiver_IMEI,Receiver_SIM_Number,Reference_Number,Merchant_Name,Merchant_Category
0,TXN0000000001,2025-01-01 00:01:21,UTR123873481368,ATM,SUCCESS,INR,12170.46,CUST0088000,IndusInd Bank,Surat,...,1596319,Verified,212481665672,NHKNJ8323P,DEV44814735,222327911347456,8930541142482852126,UTR344461950738,ATM Withdrawal,Bank
1,TXN0000000002,2025-01-01 00:01:39,UTR773663556037,Cash Deposit,SUCCESS,INR,175774.54,CUST0173395,Axis Bank,Surat,...,907364,Verified,272336379965,NBEZX3931V,DEV96392357,969917165658087,8999007851962565633,UTR630307748297,Cash Deposit,Bank
2,TXN0000000003,2025-01-01 00:03:42,UTR818079976787,Cash Deposit,SUCCESS,INR,78682.34,CUST0044406,ICICI Bank,Ahmedabad,...,1362285,Verified,329188580711,UGRDD7623J,DEV48990617,121022249994773,8992180879000828174,UTR982070254221,Cash Deposit,Bank
3,TXN0000000004,2025-01-01 00:12:03,UTR980394273515,IMPS,SUCCESS,INR,172184.82,CUST0103746,Union Bank of India,Bengaluru,...,3781561,Verified,278341675629,PCIKS6953U,DEV49560694,897359907056358,8937280074710967420,UTR886808663023,DMart,Groceries
4,TXN0000000005,2025-01-01 00:17:55,UTR572341019428,IMPS,SUCCESS,INR,1838884.55,CUST0105884,Union Bank of India,Chennai,...,2209671,Verified,575968857380,HTIIJ1627W,DEV25315074,323603801377848,8964797583753955772,UTR109933268220,Bharat Petroleum,Fuel


In [3]:
fraud_df = bank_transactions.copy()

print(fraud_df.shape)

(100000, 50)


In [4]:
fraud_df["Case_ID"] = None
fraud_df["Fraud_Type"] = "Normal"
fraud_df["Fraud_Score"] = 0.0
fraud_df["Investigation_Status"] = "Not Flagged"

In [7]:
TOTAL_TRANSACTIONS = len(fraud_df)

FRAUD_PERCENT = 0.00168

TOTAL_FRAUDS = int(TOTAL_TRANSACTIONS * FRAUD_PERCENT)

print(TOTAL_FRAUDS)

168


In [8]:
fraud_indices = random.sample(
    list(fraud_df.index),
    TOTAL_FRAUDS
)

In [9]:
fraud_types = [
    "UPI Scam",
    "Rapid Layering",
    "Mule Account",
    "ATM Cash Out",
    "SIM Swap",
    "Account Takeover",
    "Merchant Fraud",
    "Circular Money Flow"
]

In [10]:
case_number = 1

for idx in fraud_indices:

    fraud_df.loc[idx, "Case_ID"] = (
        f"CASE{case_number:05d}"
    )

    fraud_df.loc[idx, "Fraud_Type"] = random.choice(
        fraud_types
    )

    fraud_df.loc[idx, "Fraud_Score"] = round(
        random.uniform(0.80, 0.99),
        2
    )

    fraud_df.loc[idx, "Investigation_Status"] = (
        "Flagged"
    )

    case_number += 1

In [11]:
fraud_df[
    fraud_df["Fraud_Type"] != "Normal"
].head(20)

,Transaction_ID,Timestamp,UTR_Number,Transaction_Mode,Transaction_Status,Currency,Transaction_Amount,Sender_Customer_ID,Sender_Bank_Name,Sender_City,...,Receiver_Device_ID,Receiver_IMEI,Receiver_SIM_Number,Reference_Number,Merchant_Name,Merchant_Category,Case_ID,Fraud_Type,Fraud_Score,Investigation_Status
126,TXN0000000127,2025-01-01 10:44:46,UTR615481048903,UPI,SUCCESS,INR,990.44,CUST0151915,IndusInd Bank,Lucknow,...,DEV58389715,478962623548234,8961841583218918341,UTR212486324950,HP Petrol,Fuel,CASE00011,ATM Cash Out,0.90,Flagged
856,TXN0000000857,2025-01-04 02:26:40,UTR278510250227,UPI,SUCCESS,INR,12100.58,CUST0189329,ICICI Bank,Delhi,...,DEV39141774,152043455016419,8977367631890479047,UTR960642269841,Zomato,Food,CASE00075,Account Takeover,0.98,Flagged
1370,TXN0000001371,2025-01-05 22:00:16,UTR987142788988,Cash Deposit,SUCCESS,INR,29002.88,CUST0158824,IndusInd Bank,Jaipur,...,DEV86303690,146370791628859,8972543948077933701,UTR506939540907,Cash Deposit,Bank,CASE00115,Mule Account,0.84,Flagged
1516,TXN0000001517,2025-01-06 09:20:47,UTR849956987751,Cash Deposit,SUCCESS,INR,175989.41,CUST0038536,ICICI Bank,Kolkata,...,DEV15875309,361609427018882,8986639161703828175,UTR498301897400,Cash Deposit,Bank,CASE00038,UPI Scam,0.85,Flagged
4650,TXN0000004651,2025-01-17 11:10:58,UTR109795736268,UPI,PENDING,INR,22274.36,CUST0130022,Union Bank of India,Delhi,...,DEV48855432,427174226953641,8936097033825612562,UTR646981637128,Electricity Board,Utilities,CASE00153,Mule Account,0.96,Flagged
4681,TXN0000004682,2025-01-17 13:44:59,UTR532816128937,UPI,SUCCESS,INR,8176.00,CUST0081591,Canara Bank,Hyderabad,...,DEV18636784,946167913713576,8977076959114578872,UTR434801198274,IRCTC,Travel,CASE00051,Rapid Layering,0.91,Flagged
6616,TXN0000006617,2025-01-24 14:57:21,UTR399244253590,ATM,SUCCESS,INR,174305.12,CUST0108882,IndusInd Bank,Mumbai,...,DEV28027201,549248556500433,8968551240036441956,UTR628728534880,ATM Withdrawal,Bank,CASE00094,Mule Account,0.96,Flagged
7512,TXN0000007513,2025-01-27 20:48:59,UTR504329687218,Cash Deposit,SUCCESS,INR,364531.57,CUST0042933,Axis Bank,Kolkata,...,DEV69849532,184841123510836,8936642119864891492,UTR128064662766,Cash Deposit,Bank,CASE00088,Account Takeover,0.90,Flagged
7777,TXN0000007778,2025-01-28 18:02:03,UTR261367052864,UPI,SUCCESS,INR,550.04,CUST0142307,IndusInd Bank,Bhopal,...,DEV25082328,124063255371511,8929119313192306592,UTR620403762776,BigBasket,Groceries,CASE00025,Circular Money Flow,0.85,Flagged
8127,TXN0000008128,2025-01-30 00:30:52,UTR120056129071,IMPS,SUCCESS,INR,861053.99,CUST0106276,ICICI Bank,Surat,...,DEV19991387,120675355540837,8963081825158205729,UTR590597061150,Indian Oil,Fuel,CASE00080,Merchant Fraud,0.86,Flagged


In [12]:
fraud_df["Fraud_Type"].value_counts()

Fraud_Type
Normal                 99832
Account Takeover          31
SIM Swap                  23
Rapid Layering            22
Merchant Fraud            22
Mule Account              21
UPI Scam                  17
Circular Money Flow       17
ATM Cash Out              15
Name: count, dtype: int64

In [13]:
fraud_df.to_csv(
    "bank_transactions_with_cases.csv",
    index=False
)

print("Checkpoint saved.")

Checkpoint saved.


In [14]:
normal_df = fraud_df[
    fraud_df["Fraud_Type"] == "Normal"
].copy()

fraud_seed_df = fraud_df[
    fraud_df["Fraud_Type"] != "Normal"
].copy()

print("Normal :", len(normal_df))
print("Fraud Seeds :", len(fraud_seed_df))

Normal : 99832
Fraud Seeds : 168


In [15]:
generated_transactions = pd.DataFrame(columns=fraud_df.columns)

In [16]:
last_txn = fraud_df["Transaction_ID"].str.replace(
    "TXN", "", regex=False
).astype(int).max()

transaction_counter = last_txn + 1

In [17]:
last_case = fraud_seed_df["Case_ID"].str.replace(
    "CASE", "", regex=False
).astype(int).max()

case_counter = last_case + 1

In [18]:
def new_transaction_id():

    global transaction_counter

    txn = f"TXN{transaction_counter:010d}"

    transaction_counter += 1

    return txn

In [19]:
def clone_transaction(row):

    return row.copy()

In [20]:
chain_lengths = {
    "UPI Scam":4,
    "Rapid Layering":5,
    "Mule Account":6,
    "ATM Cash Out":2,
    "SIM Swap":3,
    "Account Takeover":3,
    "Merchant Fraud":2,
    "Circular Money Flow":5
}

In [21]:
fraud_seed_df = fraud_seed_df.sort_values(
    "Timestamp"
).reset_index(drop=True)

In [22]:
fraud_seed_df.head()

,Transaction_ID,Timestamp,UTR_Number,Transaction_Mode,Transaction_Status,Currency,Transaction_Amount,Sender_Customer_ID,Sender_Bank_Name,Sender_City,...,Receiver_Device_ID,Receiver_IMEI,Receiver_SIM_Number,Reference_Number,Merchant_Name,Merchant_Category,Case_ID,Fraud_Type,Fraud_Score,Investigation_Status
0,TXN0000000127,2025-01-01 10:44:46,UTR615481048903,UPI,SUCCESS,INR,990.44,CUST0151915,IndusInd Bank,Lucknow,...,DEV58389715,478962623548234,8961841583218918341,UTR212486324950,HP Petrol,Fuel,CASE00011,ATM Cash Out,0.90,Flagged
1,TXN0000000857,2025-01-04 02:26:40,UTR278510250227,UPI,SUCCESS,INR,12100.58,CUST0189329,ICICI Bank,Delhi,...,DEV39141774,152043455016419,8977367631890479047,UTR960642269841,Zomato,Food,CASE00075,Account Takeover,0.98,Flagged
2,TXN0000001371,2025-01-05 22:00:16,UTR987142788988,Cash Deposit,SUCCESS,INR,29002.88,CUST0158824,IndusInd Bank,Jaipur,...,DEV86303690,146370791628859,8972543948077933701,UTR506939540907,Cash Deposit,Bank,CASE00115,Mule Account,0.84,Flagged
3,TXN0000001517,2025-01-06 09:20:47,UTR849956987751,Cash Deposit,SUCCESS,INR,175989.41,CUST0038536,ICICI Bank,Kolkata,...,DEV15875309,361609427018882,8986639161703828175,UTR498301897400,Cash Deposit,Bank,CASE00038,UPI Scam,0.85,Flagged
4,TXN0000004651,2025-01-17 11:10:58,UTR109795736268,UPI,PENDING,INR,22274.36,CUST0130022,Union Bank of India,Delhi,...,DEV48855432,427174226953641,8936097033825612562,UTR646981637128,Electricity Board,Utilities,CASE00153,Mule Account,0.96,Flagged


In [23]:
print(fraud_seed_df["Fraud_Type"].value_counts())

Fraud_Type
Account Takeover       31
SIM Swap               23
Rapid Layering         22
Merchant Fraud         22
Mule Account           21
UPI Scam               17
Circular Money Flow    17
ATM Cash Out           15
Name: count, dtype: int64


In [25]:
import pandas as pd
import numpy as np
import random
from datetime import timedelta

# Initialize counter dynamically
all_existing_ids = pd.concat([
    normal_df["Transaction_ID"],
    fraud_seed_df["Transaction_ID"]
]).astype(str).str.extract(r'(\d+)')[0].fillna(0).astype(int)
global_txn_counter = all_existing_ids.max()

def get_new_txn_id():
    global global_txn_counter
    global_txn_counter += 1
    return f"TXN{global_txn_counter:010d}"

def get_new_utr(): return f"UTR{random.randint(100000000000, 999999999999)}"
def get_new_ref(): return f"REF{random.randint(1000000000, 9999999999)}"
def get_new_dev(): return f"DEV{random.randint(100000, 999999)}"
def get_new_imei(): return str(random.randint(100000000000000, 999999999999999))
def get_new_sim(): return f"8991{random.randint(100000000000000, 999999999999999)}"

all_cols = set(fraud_seed_df.columns)
entity_attrs = set([c.replace('Receiver_', '') for c in all_cols if c.startswith('Receiver_')] + 
                   [c.replace('Sender_', '') for c in all_cols if c.startswith('Sender_')])

banks = ["HDFC", "SBI", "ICICI", "AXIS", "KOTAK", "PNB"]
cities = ["Mumbai", "Delhi", "Bengaluru", "Kolkata", "Chennai", "Hyderabad"]

mule_pool = []
for i in range(50):
    b = random.choice(banks)
    mule_pool.append({
        'Customer_ID': f"CUST_MULE_{i:03d}", 'Name': f"Mule_Account_{i}",
        'Bank_Name': b, 'Account_Number': f"ACC{random.randint(1000000000, 9999999999)}",
        'IFSC': f"{b[:4]}000{random.randint(1000, 9999)}", 'UPI_ID': f"mule{i}@{b.lower()}",
        'Phone_Number': f"9{random.randint(100000000, 999999999)}", 'Email': f"mule{i}@proxy.com",
        'City': random.choice(cities), 'State': "State_Placeholder", 'PIN': f"{random.randint(110000, 899999)}",
        'DOB': "1995-01-01", 'Gender': random.choice(["M", "F"]), 'Occupation': "Student",
        'Annual_Income': "Below 1L", 'KYC_Status': "Verified", 'Aadhaar': str(random.randint(100000000000, 999999999999)),
        'PAN': f"ABCDE{random.randint(1000, 9999)}F", 'Device_ID': f"DEV_MULE_{i}",
        'IMEI': get_new_imei(), 'SIM_Number': get_new_sim()
    })

merchant_pool = []
for i in range(50):
    b = random.choice(banks)
    merchant_pool.append({
        'Customer_ID': f"MERCH_FAKE_{i:03d}", 'Name': f"Fake_Store_{i}",
        'Bank_Name': b, 'Account_Number': f"MERCH{random.randint(1000000000, 9999999999)}",
        'IFSC': f"{b[:4]}000{random.randint(1000, 9999)}", 'UPI_ID': f"fakestore{i}@paytm",
        'Phone_Number': f"8{random.randint(100000000, 999999999)}", 'Email': f"support@fakestore{i}.com",
        'City': random.choice(cities), 'State': "State_Placeholder", 'PIN': f"{random.randint(110000, 899999)}",
        'DOB': "1985-01-01", 'Gender': "Company", 'Occupation': "Retail",
        'Annual_Income': "10L-50L", 'KYC_Status': "Verified", 'Aadhaar': "NA",
        'PAN': f"MERCH{random.randint(1000, 9999)}Z", 'Device_ID': f"DEV_MERCH_{i}",
        'IMEI': get_new_imei(), 'SIM_Number': get_new_sim()
    })

def get_atm(bank_name):
    return {
        'Customer_ID': 'ATM_CASH_OUT', 'Name': f"ATM_{bank_name}_{random.randint(100,999)}",
        'Bank_Name': bank_name, 'Account_Number': 'CASH_WDL', 'IFSC': 'NA', 'UPI_ID': 'NA',
        'Phone_Number': 'NA', 'Email': 'NA', 'City': 'NA', 'State': 'NA', 'PIN': 'NA',
        'DOB': 'NA', 'Gender': 'NA', 'Occupation': 'NA', 'Annual_Income': 'NA',
        'KYC_Status': 'NA', 'Aadhaar': 'NA', 'PAN': 'NA', 'Device_ID': f"ATM_TERM_{random.randint(1000,9999)}",
        'IMEI': 'NA', 'SIM_Number': 'NA'
    }

def get_victim_receiver(seed):
    victim = {}
    for attr in entity_attrs:
        snd_col = f"Sender_{attr}"
        if snd_col in seed:
            victim[attr] = seed[snd_col]
    return victim

def create_hop(prev_txn, receiver_info, mode, amount_factor, current_time):
    new_txn = prev_txn.copy()
    if 'Transaction_ID' in new_txn: new_txn['Transaction_ID'] = get_new_txn_id()
    if 'UTR_Number' in new_txn: new_txn['UTR_Number'] = get_new_utr()
    if 'Reference_Number' in new_txn: new_txn['Reference_Number'] = get_new_ref()
    
    new_time = current_time + timedelta(seconds=random.randint(30, 600))
    if 'Timestamp' in new_txn: new_txn['Timestamp'] = new_time.strftime('%Y-%m-%d %H:%M:%S')
    
    amt = round(float(prev_txn['Transaction_Amount']) * amount_factor, 2)
    new_txn['Transaction_Amount'] = amt
    if 'Transaction_Mode' in new_txn: new_txn['Transaction_Mode'] = mode
    
    # Balance Updates
    balance_maps = [
        ('Sender_Old_Balance', 'Sender_New_Balance'),
        ('Receiver_Old_Balance', 'Receiver_New_Balance'),
        ('oldbalanceOrg', 'newbalanceOrig'),
        ('oldbalanceDest', 'newbalanceDest')
    ]
    for old_col, new_col in balance_maps:
        if old_col in new_txn and new_col in new_txn:
            if old_col.startswith('Sender') or old_col == 'oldbalanceOrg':
                new_txn[old_col] = prev_txn.get(new_col, 0)
                new_txn[new_col] = float(new_txn[old_col]) - amt
            else: 
                new_txn[old_col] = prev_txn.get(new_col, 0)
                new_txn[new_col] = float(new_txn[old_col]) + amt

    for attr in entity_attrs:
        rcv_col = f"Receiver_{attr}"
        snd_col = f"Sender_{attr}"
        if rcv_col in prev_txn and snd_col in new_txn:
            new_txn[snd_col] = prev_txn[rcv_col]
        if attr in receiver_info and rcv_col in new_txn:
            new_txn[rcv_col] = receiver_info[attr]
            
    if mode in ['POS', 'RETAIL']:
        if 'Merchant_Name' in new_txn: new_txn['Merchant_Name'] = receiver_info.get('Name', 'Fake Store')
        if 'Merchant_Category' in new_txn: new_txn['Merchant_Category'] = 'RETAIL'
    elif mode == 'CASH':
        if 'Merchant_Name' in new_txn: new_txn['Merchant_Name'] = 'NA'
        if 'Merchant_Category' in new_txn: new_txn['Merchant_Category'] = 'ATM'
    else:
        if 'Merchant_Name' in new_txn: new_txn['Merchant_Name'] = 'NA'
        if 'Merchant_Category' in new_txn: new_txn['Merchant_Category'] = 'Bank Transfer'
        
    if 'Fraud_Score' in new_txn:
        try:
            current_score = float(new_txn['Fraud_Score'])
            if pd.isna(current_score): current_score = 50.0
        except (ValueError, TypeError):
            current_score = 50.0
        new_txn['Fraud_Score'] = min(100.0, current_score + random.uniform(1.0, 5.0))
        
    return new_txn, new_time

clean_normal_df = normal_df[~normal_df['Transaction_ID'].isin(fraud_seed_df['Transaction_ID'])].copy()
modified_seeds = []
generated_txns = []
ground_truth = []
case_idx = 1

for _, row in fraud_seed_df.iterrows():
    seed_txn = row.to_dict()
    fraud_type = seed_txn.get('Fraud_Type', 'UPI Scam')
    
    if isinstance(seed_txn.get('Timestamp'), str):
        seed_time = pd.to_datetime(seed_txn['Timestamp'])
    elif pd.notna(seed_txn.get('Timestamp')):
        seed_time = seed_txn['Timestamp']
        seed_txn['Timestamp'] = seed_time.strftime('%Y-%m-%d %H:%M:%S')
    else:
        seed_time = pd.Timestamp.now()

    if fraud_type in ['SIM Swap', 'Account Takeover']:
        if 'Sender_Device_ID' in seed_txn: seed_txn['Sender_Device_ID'] = get_new_dev()
        if 'Sender_IMEI' in seed_txn: seed_txn['Sender_IMEI'] = get_new_imei()
        if 'Sender_SIM_Number' in seed_txn: seed_txn['Sender_SIM_Number'] = get_new_sim()
        
    if fraud_type == 'Merchant Fraud':
        fake_merchant = random.choice(merchant_pool)
        for attr in entity_attrs:
            rcv_col = f"Receiver_{attr}"
            if rcv_col in seed_txn and attr in fake_merchant:
                seed_txn[rcv_col] = fake_merchant[attr]
        if 'Transaction_Mode' in seed_txn: seed_txn['Transaction_Mode'] = 'POS'
        if 'Merchant_Category' in seed_txn: seed_txn['Merchant_Category'] = 'RETAIL'
        if 'Merchant_Name' in seed_txn: seed_txn['Merchant_Name'] = fake_merchant.get('Name', 'Fake Store')

    if seed_txn.get('Merchant_Category') not in ['RETAIL', 'ATM', 'POS']:
        if 'Merchant_Name' in seed_txn: seed_txn['Merchant_Name'] = 'NA'
        if 'Merchant_Category' in seed_txn: seed_txn['Merchant_Category'] = 'Bank Transfer'
        
    modified_seeds.append(seed_txn)
    current_txn = seed_txn.copy()
    current_time = seed_time
    chain_ids = [seed_txn.get('Transaction_ID', 'UNKNOWN')]
    
    if fraud_type == 'UPI Scam':
        h1, current_time = create_hop(current_txn, random.choice(mule_pool), 'UPI', random.uniform(0.98, 0.99), current_time)
        chain_ids.append(h1.get('Transaction_ID', '')); generated_txns.append(h1); current_txn = h1
        h2, current_time = create_hop(current_txn, get_atm(current_txn.get('Receiver_Bank_Name', 'SBI')), 'CASH', random.uniform(0.98, 0.99), current_time)
        chain_ids.append(h2.get('Transaction_ID', '')); generated_txns.append(h2); current_txn = h2
        
    elif fraud_type == 'Rapid Layering':
        h1, current_time = create_hop(current_txn, random.choice(mule_pool), 'IMPS', random.uniform(0.65, 0.75), current_time)
        chain_ids.append(h1.get('Transaction_ID', '')); generated_txns.append(h1); current_txn = h1
        h2, current_time = create_hop(current_txn, random.choice(mule_pool), 'NEFT', random.uniform(0.40, 0.60), current_time)
        chain_ids.append(h2.get('Transaction_ID', '')); generated_txns.append(h2); current_txn = h2
        h3, current_time = create_hop(current_txn, random.choice(mule_pool), 'RTGS', random.uniform(0.80, 0.90), current_time)
        chain_ids.append(h3.get('Transaction_ID', '')); generated_txns.append(h3); current_txn = h3
        h4, current_time = create_hop(current_txn, get_atm(current_txn.get('Receiver_Bank_Name', 'SBI')), 'CASH', random.uniform(0.98, 0.99), current_time)
        chain_ids.append(h4.get('Transaction_ID', '')); generated_txns.append(h4); current_txn = h4
        
    elif fraud_type == 'Mule Account':
        h1, current_time = create_hop(current_txn, random.choice(mule_pool), 'IMPS', random.uniform(0.95, 0.98), current_time)
        chain_ids.append(h1.get('Transaction_ID', '')); generated_txns.append(h1); current_txn = h1
        h2, current_time = create_hop(current_txn, random.choice(mule_pool), 'IMPS', random.uniform(0.95, 0.98), current_time)
        chain_ids.append(h2.get('Transaction_ID', '')); generated_txns.append(h2); current_txn = h2
        
    elif fraud_type == 'ATM Cash Out' or fraud_type == 'SIM Swap':
        h1, current_time = create_hop(current_txn, get_atm(current_txn.get('Receiver_Bank_Name', 'SBI')), 'CASH', random.uniform(0.98, 0.99), current_time)
        chain_ids.append(h1.get('Transaction_ID', '')); generated_txns.append(h1); current_txn = h1
        
    elif fraud_type == 'Account Takeover':
        h1, current_time = create_hop(current_txn, random.choice(merchant_pool), 'POS', random.uniform(0.98, 0.99), current_time)
        chain_ids.append(h1.get('Transaction_ID', '')); generated_txns.append(h1); current_txn = h1
        
    elif fraud_type == 'Circular Money Flow':
        h1, current_time = create_hop(current_txn, random.choice(mule_pool), 'IMPS', random.uniform(0.96, 0.98), current_time)
        chain_ids.append(h1.get('Transaction_ID', '')); generated_txns.append(h1); current_txn = h1
        h2, current_time = create_hop(current_txn, random.choice(mule_pool), 'IMPS', random.uniform(0.96, 0.98), current_time)
        chain_ids.append(h2.get('Transaction_ID', '')); generated_txns.append(h2); current_txn = h2
        h3, current_time = create_hop(current_txn, get_victim_receiver(seed_txn), 'IMPS', random.uniform(0.96, 0.98), current_time)
        chain_ids.append(h3.get('Transaction_ID', '')); generated_txns.append(h3); current_txn = h3

    ground_truth.append({
        'Case_ID': seed_txn.get('Case_ID', f"CASE_{case_idx:05d}"),
        'Fraud_Type': fraud_type,
        'Victim_Account': seed_txn.get('Sender_Account_Number', 'UNKNOWN'),
        'Scammer_Account': seed_txn.get('Receiver_Account_Number', 'UNKNOWN'),
        'Chain_Length': len(chain_ids),
        'Case_Start_Time': seed_time.strftime('%Y-%m-%d %H:%M:%S'),
        'Case_End_Time': current_time.strftime('%Y-%m-%d %H:%M:%S'),
        'All_Transaction_IDs': "|".join(chain_ids),
        'Final_Cashout_Account': current_txn.get('Receiver_Account_Number', 'UNKNOWN')
    })
    case_idx += 1

final_banking_df = pd.concat([
    clean_normal_df, 
    pd.DataFrame(modified_seeds), 
    pd.DataFrame(generated_txns)
], ignore_index=True)

if 'Timestamp' in final_banking_df.columns:
    final_banking_df['Timestamp'] = pd.to_datetime(final_banking_df['Timestamp'])
    final_banking_df.sort_values(by='Timestamp', ascending=True, inplace=True)

final_banking_df.reset_index(drop=True, inplace=True)
ground_truth_df = pd.DataFrame(ground_truth)

final_banking_df.to_csv('final_bank_transactions_with_fraud.csv', index=False)
ground_truth_df.to_csv('fraud_ground_truth.csv', index=False)

In [26]:
print(final_banking_df["Fraud_Type"].value_counts(dropna=False))

Fraud_Type
Normal                 99832
Rapid Layering           110
Circular Money Flow       68
Mule Account              63
Account Takeover          62
UPI Scam                  51
SIM Swap                  46
ATM Cash Out              30
Merchant Fraud            22
Name: count, dtype: int64


In [27]:
print(len(ground_truth_df))

168


In [28]:
ground_truth_df["Chain_Length"].value_counts().sort_index()

Chain_Length
1    22
2    69
3    38
4    17
5    22
Name: count, dtype: int64

In [29]:
ground_truth_df.groupby("Fraud_Type")["Chain_Length"].value_counts()

Fraud_Type           Chain_Length
ATM Cash Out         2               15
Account Takeover     2               31
Circular Money Flow  4               17
Merchant Fraud       1               22
Mule Account         3               21
Rapid Layering       5               22
SIM Swap             2               23
UPI Scam             3               17
Name: count, dtype: int64

In [30]:
## sanity checks:


In [31]:
# 1. Duplicate Transaction IDs
print(final_banking_df["Transaction_ID"].duplicated().sum())

0


In [32]:
# 2. Missing Transaction IDs
print(final_banking_df["Transaction_ID"].isna().sum())

0


In [33]:
# 3. Ground truth consistency
print(ground_truth_df["Chain_Length"].sum())
print((final_banking_df["Fraud_Type"]!="Normal").sum())

452
452


In [34]:
for i, col in enumerate(final_banking_df.columns, start=1):
    print(f"{i}. {col}")

1. Transaction_ID
2. Timestamp
3. UTR_Number
4. Transaction_Mode
5. Transaction_Status
6. Currency
7. Transaction_Amount
8. Sender_Customer_ID
9. Sender_Bank_Name
10. Sender_City
11. Sender_Account_Number
12. Sender_IFSC
13. Sender_Phone_Number
14. Sender_UPI_ID
15. Sender_Gender
16. Sender_Customer_Name
17. Sender_Email
18. Sender_DOB
19. Sender_Customer_Since
20. Sender_Occupation
21. Sender_Annual_Income
22. Sender_KYC_Status
23. Sender_Aadhaar
24. Sender_PAN
25. Sender_Device_ID
26. Sender_IMEI
27. Sender_SIM_Number
28. Receiver_Customer_ID
29. Receiver_Bank_Name
30. Receiver_City
31. Receiver_Account_Number
32. Receiver_IFSC
33. Receiver_Phone_Number
34. Receiver_UPI_ID
35. Receiver_Gender
36. Receiver_Customer_Name
37. Receiver_Email
38. Receiver_DOB
39. Receiver_Customer_Since
40. Receiver_Occupation
41. Receiver_Annual_Income
42. Receiver_KYC_Status
43. Receiver_Aadhaar
44. Receiver_PAN
45. Receiver_Device_ID
46. Receiver_IMEI
47. Receiver_SIM_Number
48. Reference_Number
49. Me

In [35]:
## making necesary changes for the dataset:

In [38]:
import pandas as pd

# Load dataset
df = pd.read_csv("final_bank_transactions_with_fraud.csv")

print("="*80)
print("CUSTOMER CONSISTENCY CHECK")
print("="*80)

# ---------------------------------------------------------
# Helper Function
# ---------------------------------------------------------
def check_consistency(customer_col, attribute_col):
    grouped = df.groupby(customer_col)[attribute_col].nunique(dropna=False)
    inconsistent = grouped[grouped > 1]

    print(f"\nChecking {customer_col} --> {attribute_col}")

    if len(inconsistent) == 0:
        print("✅ PASSED")
        print("No inconsistencies found.")
    else:
        print(f"❌ FAILED ({len(inconsistent)} inconsistent customers)")
        print("\nSample Customer IDs:")
        print(inconsistent.head(10))

        print("\nCorresponding values:")
        for cid in inconsistent.head(5).index:
            print(f"\nCustomer ID: {cid}")
            print(
                df[df[customer_col] == cid][[customer_col, attribute_col]]
                .drop_duplicates()
            )

# =========================================================
# SENDER CHECKS
# =========================================================

check_consistency("Sender_Customer_ID", "Sender_Customer_Name")
check_consistency("Sender_Customer_ID", "Sender_Phone_Number")
check_consistency("Sender_Customer_ID", "Sender_DOB")
check_consistency("Sender_Customer_ID", "Sender_Email")
check_consistency("Sender_Customer_ID", "Sender_Account_Number")
check_consistency("Sender_Customer_ID", "Sender_IFSC")
check_consistency("Sender_Customer_ID", "Sender_Bank_Name")

# =========================================================
# RECEIVER CHECKS
# =========================================================

check_consistency("Receiver_Customer_ID", "Receiver_Customer_Name")
check_consistency("Receiver_Customer_ID", "Receiver_Phone_Number")
check_consistency("Receiver_Customer_ID", "Receiver_DOB")
check_consistency("Receiver_Customer_ID", "Receiver_Email")
check_consistency("Receiver_Customer_ID", "Receiver_Account_Number")
check_consistency("Receiver_Customer_ID", "Receiver_IFSC")
check_consistency("Receiver_Customer_ID", "Receiver_Bank_Name")

print("\n" + "="*80)
print("CHECK COMPLETE")
print("="*80)

CUSTOMER CONSISTENCY CHECK

Checking Sender_Customer_ID --> Sender_Customer_Name
❌ FAILED (36 inconsistent customers)

Sample Customer IDs:
Sender_Customer_ID
CUST_MULE_003    3
CUST_MULE_004    4
CUST_MULE_005    2
CUST_MULE_006    4
CUST_MULE_007    3
CUST_MULE_010    2
CUST_MULE_012    3
CUST_MULE_014    5
CUST_MULE_015    3
CUST_MULE_016    4
Name: Sender_Customer_Name, dtype: int64

Corresponding values:

Customer ID: CUST_MULE_003
      Sender_Customer_ID Sender_Customer_Name
57685      CUST_MULE_003       Kartik Solanki
63066      CUST_MULE_003          Indira Negi
84141      CUST_MULE_003       Devansh Shinde

Customer ID: CUST_MULE_004
      Sender_Customer_ID Sender_Customer_Name
37810      CUST_MULE_004     Kalyani Deshmukh
40677      CUST_MULE_004        Aashi Mahajan
56506      CUST_MULE_004          Ananya Bora
81812      CUST_MULE_004       Reyansh Tiwari

Customer ID: CUST_MULE_005
      Sender_Customer_ID Sender_Customer_Name
16432      CUST_MULE_005           Indira R

In [39]:
import pandas as pd
import numpy as np
import random
import re
import os

# ==========================================
# SETUP & CONFIGURATION
# ==========================================
FILE_PATH = "final_bank_transactions_with_fraud.csv"
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

def main():
    if not os.path.exists(FILE_PATH):
        print(f"Error: The file {FILE_PATH} does not exist in the current directory.")
        return

    # Load original data
    print("Loading existing dataset...")
    df = pd.read_csv(FILE_PATH)
    
    # Store original state for validation
    original_shape = df.shape
    original_columns = df.columns.tolist()
    original_transaction_ids = df['Transaction_ID'].copy()
    original_fraud_labels = df[['Case_ID', 'Fraud_Type', 'Fraud_Score', 'Investigation_Status']].copy()

    # ==========================================
    # MODIFICATION 1: Robust Merchant Cleanup
    # ==========================================
    print("1. Cleaning up merchant data (using robust regex matching)...")
    if 'Transaction_Mode' in df.columns:
        # Case-insensitive substring matching to handle variations like "ATM CASH WITHDRAWAL"
        pattern = r'atm|cash\s*withdraw|cash\s*deposit|self\s*transfer|balance\s*transfer'
        mask = df['Transaction_Mode'].str.contains(pattern, case=False, na=False)
        
        if 'Merchant_Name' in df.columns:
            df.loc[mask, 'Merchant_Name'] = np.nan
        if 'Merchant_Category' in df.columns:
            df.loc[mask, 'Merchant_Category'] = np.nan

    # ==========================================
    # MODIFICATION 2: Add Channel Column
    # ==========================================
    print("2. Adding Channel column...")
    def determine_channel(mode):
        mode = str(mode).upper()
        if "UPI" in mode:
            return "Mobile_App"
        elif "ATM" in mode or "WITHDRAWAL" in mode:
            return "ATM"
        elif "CASH DEPOSIT" in mode or "BRANCH" in mode:
            return "Branch"
        elif mode in ["NEFT", "RTGS"]:
            return "Internet_Banking"
        elif "IMPS" in mode:
            return np.random.choice(["Mobile_App", "Internet_Banking"], p=[0.7, 0.3])
        elif "POS" in mode or "CARD" in mode:
            return "POS"
        else:
            return np.random.choice(["Mobile_App", "Internet_Banking", "Branch"], p=[0.6, 0.3, 0.1])

    df['Channel'] = df['Transaction_Mode'].apply(determine_channel)

    # ==========================================
    # MODIFICATION 3, 4, 7, 8, 9: Customer-Level Consistency
    # Generate Account Type, UPI, and Email ONCE per Customer
    # ==========================================
    print("3, 5, 7-9. Generating stable Customer Profiles (Account Type, UPI, Email)...")
    
    # Extract unique customers from both sender and receiver columns
    senders = df[['Sender_Customer_ID', 'Sender_Customer_Name', 'Sender_Phone_Number']].rename(
        columns={'Sender_Customer_ID': 'Customer_ID', 'Sender_Customer_Name': 'Name', 'Sender_Phone_Number': 'Phone'}
    )
    receivers = df[['Receiver_Customer_ID', 'Receiver_Customer_Name', 'Receiver_Phone_Number']].rename(
        columns={'Receiver_Customer_ID': 'Customer_ID', 'Receiver_Customer_Name': 'Name', 'Receiver_Phone_Number': 'Phone'}
    )
    
    # Combine all customer records
    all_customers = pd.concat([senders, receivers]).dropna(subset=['Customer_ID'])
    
    # Build a canonical name mapping (first non-null name encountered)
    canonical_names = all_customers.dropna(subset=['Name']).drop_duplicates(subset=['Customer_ID']).set_index('Customer_ID')['Name']
    
    # Get unique customers dataframe
    customers = all_customers.drop_duplicates(subset=['Customer_ID']).copy()
    
    # Apply the canonical name mapping to the temporary customers dataframe
    customers['Name'] = customers['Customer_ID'].map(canonical_names)
    
    # Generate Account Type (Stable per customer)
    account_types = ["Savings", "Current", "Salary", "Demat"]
    probs = [0.70, 0.15, 0.13, 0.02]
    customers['Account_Type'] = np.random.choice(account_types, size=len(customers), p=probs)
    
    # Advanced Name Processing for UPI and Email
    upi_domains = ['@oksbi', '@ybl', '@ibl', '@okaxis', '@okhdfcbank', '@paytm', '@axl', '@apl']
    email_domains = ['@gmail.com', '@yahoo.com', '@outlook.com', '@icloud.com', '@proton.me', '@rediffmail.com']

    def clean_and_split_name(name):
        if pd.isna(name): return ["user"]
        # Keep only alphanumeric and space
        name = re.sub(r'[^a-zA-Z0-9\s]', '', str(name)).lower().strip()
        parts = name.split()
        return parts if parts else ["user"]

    def generate_upi_email(row):
        parts = clean_and_split_name(row['Name'])
        f = parts[0]
        l = parts[1] if len(parts) > 1 else ""
        
        # Format Phone Number
        phone_str = str(row['Phone']).replace('.0', '') if pd.notna(row['Phone']) else ""
        if len(phone_str) < 10:
            phone_str = str(random.randint(9000000000, 9999999999))
            
        # Realistic username patterns
        username_patterns = [
            f,
            f"{f}.{l}" if l else f,
            f"{f}{l[0]}" if l else f,
            f"{f}{random.randint(10,999)}",
            f"{f[0]}.{l}" if l else f,
            f"{f}_work",
            f"{f}{l[0]}{random.randint(10,99)}" if l else f
        ]
        
        email_base = random.choice(username_patterns).replace('..', '.').strip('.')
        email = f"{email_base}{random.choice(email_domains)}"
        
        upi_patterns = [
            phone_str[-10:], # phone-based
            email_base.replace('.', '').replace('_', ''), # name-based
            f"{f}{random.randint(10,999)}" # initial/name + number
        ]
        upi_base = random.choice(upi_patterns)
        upi_id = f"{upi_base}{random.choice(upi_domains)}"
        
        return upi_id, email

    # Apply generation to unique customers
    customers[['UPI_ID', 'Email']] = customers.apply(generate_upi_email, axis=1, result_type='expand')
    
    # Map back to the main dataframe
    acct_map = customers.set_index('Customer_ID')['Account_Type'].to_dict()
    upi_map = customers.set_index('Customer_ID')['UPI_ID'].to_dict()
    email_map = customers.set_index('Customer_ID')['Email'].to_dict()
    
    df['Sender_Account_Type'] = df['Sender_Customer_ID'].map(acct_map)
    df['Sender_UPI_ID'] = df['Sender_Customer_ID'].map(upi_map)
    df['Sender_Email'] = df['Sender_Customer_ID'].map(email_map)
    
    df['Receiver_Account_Type'] = df['Receiver_Customer_ID'].map(acct_map)
    df['Receiver_UPI_ID'] = df['Receiver_Customer_ID'].map(upi_map)
    df['Receiver_Email'] = df['Receiver_Customer_ID'].map(email_map)

    # ==========================================
    # MODIFICATION 4 & 10: Monthly Salary as Integers
    # ==========================================
    print("4 & 10. Replacing Annual Income with Monthly Salary (Rounded to Integers)...")
    
    def clean_currency_to_numeric(series):
        if series.dtype == object:
            cleaned = series.astype(str).str.replace(r'[^\d.]', '', regex=True)
            return pd.to_numeric(cleaned, errors='coerce')
        return pd.to_numeric(series, errors='coerce')

    if 'Sender_Annual_Income' in df.columns:
        df['Sender_Annual_Income'] = clean_currency_to_numeric(df['Sender_Annual_Income'])
        # Round to whole number and use Int64 to allow NaN support for integers
        df['Sender_Monthly_Salary'] = (df['Sender_Annual_Income'] / 12).round(0).astype('Int64')
        df.drop(columns=['Sender_Annual_Income'], inplace=True)
        
    if 'Receiver_Annual_Income' in df.columns:
        df['Receiver_Annual_Income'] = clean_currency_to_numeric(df['Receiver_Annual_Income'])
        df['Receiver_Monthly_Salary'] = (df['Receiver_Annual_Income'] / 12).round(0).astype('Int64')
        df.drop(columns=['Receiver_Annual_Income'], inplace=True)

    # ==========================================
    # MODIFICATION 7: Missing City Values
    # ==========================================
    print("7. Introducing 5-10% missing city values...")
    def introduce_nans(col_data, pct=0.08):
        mask = np.random.rand(len(col_data)) < pct
        col_data.loc[mask] = np.nan
        return col_data

    if 'Sender_City' in df.columns:
        df['Sender_City'] = introduce_nans(df['Sender_City'].copy(), pct=random.uniform(0.05, 0.10))
    if 'Receiver_City' in df.columns:
        df['Receiver_City'] = introduce_nans(df['Receiver_City'].copy(), pct=random.uniform(0.05, 0.10))

    # ==========================================
    # MODIFICATION 6: Improve Transaction Amount Logic
    # ==========================================
    print("6. Overwriting Transaction Amounts with Mode-Aware realistic distribution...")
    
    def generate_amount(mode):
        mode = str(mode).upper()
        p = np.random.rand()
        
        if any(x in mode for x in ["ATM", "WITHDRAWAL"]):
            # ATM: Max ~40,000 realistically
            if p < 0.95: return round(np.random.uniform(100, 20000), 2)
            else: return round(np.random.uniform(20000, 40000), 2)
            
        elif "CASH DEPOSIT" in mode:
            # Cash Deposit: Occasional large deposits
            if p < 0.85: return round(np.random.uniform(500, 50000), 2)
            else: return round(np.random.uniform(50000, 200000), 2)
            
        elif "UPI" in mode:
            # UPI: mostly smaller values
            if p < 0.98: return round(np.random.uniform(50, 25000), 2)
            else: return round(np.random.uniform(25000, 100000), 2)
            
        elif "IMPS" in mode:
            # IMPS: Limit is generally up to 5 Lakhs, mostly smaller
            if p < 0.90: return round(np.random.uniform(500, 50000), 2)
            else: return round(np.random.uniform(50000, 200000), 2)
            
        elif "RTGS" in mode:
            # RTGS: Minimum 2 Lakh, goes up to very high but controlled
            if p < 0.95: return round(np.random.uniform(200000, 1500000), 2)
            else: return round(np.random.uniform(1500000, 5000000), 2)
            
        else: # NEFT, POS, etc.
            # Wide distribution
            if p < 0.80: return round(np.random.uniform(100, 25000), 2)
            elif p < 0.95: return round(np.random.uniform(25000, 200000), 2)
            else: return round(np.random.uniform(200000, 1000000), 2)

    if 'Transaction_Mode' in df.columns:
        df['Transaction_Amount'] = df['Transaction_Mode'].apply(generate_amount)
    else:
        df['Transaction_Amount'] = [generate_amount("UNKNOWN") for _ in range(len(df))]

    # ==========================================
    # FINAL VALIDATIONS
    # ==========================================
    print("\n--- Running Validations ---")
    
    # 1. Row & Core Column Integrity
    assert df.shape[0] == original_shape[0], "Row count mismatch! Rows were added or removed."
    assert df['Transaction_ID'].equals(original_transaction_ids), "Transaction_ID values were altered or shuffled!"
    for col in ['Case_ID', 'Fraud_Type', 'Fraud_Score', 'Investigation_Status']:
        assert df[col].equals(original_fraud_labels[col]), f"{col} column was altered!"
        
    # 2. Customer-Level Consistency Constraints
    print("Validating Customer-Level Consistency...")
    for prefix in ['Sender', 'Receiver']:
        # Dropna=True inside nunique ensures we don't count NaN as a distinct second value if a row missed a mapping somehow
        acct_unique = df.groupby(f'{prefix}_Customer_ID')[f'{prefix}_Account_Type'].nunique().max()
        upi_unique = df.groupby(f'{prefix}_Customer_ID')[f'{prefix}_UPI_ID'].nunique().max()
        email_unique = df.groupby(f'{prefix}_Customer_ID')[f'{prefix}_Email'].nunique().max()
        
        assert acct_unique == 1, f"Failed: A {prefix} customer has multiple Account Types!"
        assert upi_unique == 1, f"Failed: A {prefix} customer has multiple UPI IDs!"
        assert email_unique == 1, f"Failed: A {prefix} customer has multiple Emails!"

    print("[VALIDATION SUCCESSFUL] All constraints and consistencies upheld.")
    
    # Stats Calculation
    amount_col = df['Transaction_Amount']
    pct_under_25k = (amount_col <= 25000).mean() * 100
    pct_25k_to_5l = ((amount_col > 25000) & (amount_col <= 500000)).mean() * 100
    pct_over_5l = (amount_col > 500000).mean() * 100
    sender_city_nans = df['Sender_City'].isna().sum()
    receiver_city_nans = df['Receiver_City'].isna().sum()

    # ==========================================
    # SAVE & REPORT
    # ==========================================
    print(f"\nSaving modified file over {FILE_PATH}...")
    df.to_csv(FILE_PATH, index=False)
    
    print("\n==========================================")
    print("          TRANSFORMATION SUMMARY          ")
    print("==========================================")
    print(f"Original dataframe shape: {original_shape}")
    print(f"Final dataframe shape:    {df.shape}")
    
    print("\n[Missing City Values Introduced]:")
    print(f"- Sender_City: {sender_city_nans} ({(sender_city_nans/len(df))*100:.2f}%)")
    print(f"- Receiver_City: {receiver_city_nans} ({(receiver_city_nans/len(df))*100:.2f}%)")
    
    print("\n[Transaction Amount Distribution Metrics]:")
    print(f"- Under ₹25,000 (Target ~90%): {pct_under_25k:.2f}%")
    print(f"- ₹25,000 - ₹5,00,000 (Target ~9%): {pct_25k_to_5l:.2f}%")
    print(f"- Above ₹5,00,000 (Target ~1%): {pct_over_5l:.2f}%")
    
    print("\nConfirmation: CUSTOMER PROFILES ARE STABLE. NO ROWS ADDED/REMOVED. LABELS PRESERVED.")

if __name__ == "__main__":
    main()

Loading existing dataset...
1. Cleaning up merchant data (using robust regex matching)...
2. Adding Channel column...
3, 5, 7-9. Generating stable Customer Profiles (Account Type, UPI, Email)...
4 & 10. Replacing Annual Income with Monthly Salary (Rounded to Integers)...
7. Introducing 5-10% missing city values...
6. Overwriting Transaction Amounts with Mode-Aware realistic distribution...

--- Running Validations ---
Validating Customer-Level Consistency...
[VALIDATION SUCCESSFUL] All constraints and consistencies upheld.

Saving modified file over final_bank_transactions_with_fraud.csv...

          TRANSFORMATION SUMMARY          
Original dataframe shape: (100284, 54)
Final dataframe shape:    (100284, 57)

[Missing City Values Introduced]:
- Sender_City: 7072 (7.05%)
- Receiver_City: 9814 (9.79%)

[Transaction Amount Distribution Metrics]:
- Under ₹25,000 (Target ~90%): 80.63%
- ₹25,000 - ₹5,00,000 (Target ~9%): 19.33%
- Above ₹5,00,000 (Target ~1%): 0.04%

Confirmation: CUSTOMER 